# Bienvenidos
Este JupiterNoteBook esta creado para:
1. Arreglar las fotos y videos que tengan una corrupcion minima
2. Organizarlos por fecha
3. Mejorarles la calidad

## 1. Arreglar Fotos y Videos

In [ ]:
import os
import sys
import cv2
from PIL import Image, ImageFile
from concurrent.futures import ProcessPoolExecutor
import subprocess

# Habilitar carga de imágenes truncadas para mayor robustez
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Extensiones de archivos soportados
IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp']
VIDEO_EXTENSIONS = ['.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv']

def check_ffmpeg():
    """Verifica si FFmpeg está instalado."""
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

def check_imagemagick():
    """Verifica si ImageMagick está instalado."""
    try:
        result = subprocess.run(['magick', '-version'], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

def repair_image(file_path, output_dir):
    """Intenta reparar una imagen usando Pillow con modos robustos, luego ImageMagick."""
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    
    # Intentar abrir y guardar directamente (para imágenes truncadas)
    try:
        with Image.open(file_path) as img:
            img.save(output_path, quality=95)
        print(f"Imagen reparada con Pillow (directo): {output_path}")
        return
    except Exception as e:
        print(f"Pillow directo falló para {file_path}: {e}")
    
    # Intentar con verify
    try:
        with Image.open(file_path) as img:
            img.verify()
            img.close()
            with Image.open(file_path) as img:
                img.save(output_path, quality=95)
        print(f"Imagen reparada con Pillow (verify): {output_path}")
        return
    except Exception as e:
        print(f"Pillow verify falló para {file_path}: {e}")

    # Fallback a ImageMagick
    if check_imagemagick():
        cmd = ['magick', 'convert', file_path, output_path]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            if result.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                print(f"Imagen reparada con ImageMagick: {output_path}")
                return
            else:
                print(f"ImageMagick falló para {file_path}: {result.stderr}")
        except subprocess.TimeoutExpired:
            print(f"Timeout con ImageMagick para {file_path}")
        except Exception as e:
            print(f"Error con ImageMagick en {file_path}: {e}")
    
    print(f"No se pudo reparar la imagen {file_path} (demasiado corrupta)")

def repair_video_ffmpeg(file_path, output_dir):
    """Repara video usando FFmpeg: intenta copy primero, luego re-encode si falla."""
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    
    # Intentar copy primero (más rápido, preserva calidad)
    cmd_copy = [
        'ffmpeg',
        '-fflags', '+discardcorrupt',
        '-err_detect', 'ignore_err',
        '-i', file_path,
        '-c', 'copy',
        '-y',
        output_path
    ]
    try:
        result = subprocess.run(cmd_copy, capture_output=True, text=True, timeout=300)
        if result.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print(f"Video reparado con FFmpeg (copy): {output_path}")
            return True
    except subprocess.TimeoutExpired:
        print(f"Timeout en copy para {file_path}")
    except Exception as e:
        print(f"Error en copy para {file_path}: {e}")

    # Si copy falla, intentar re-encode (más lento, pero repara más)
    cmd_encode = [
        'ffmpeg',
        '-fflags', '+discardcorrupt',
        '-err_detect', 'ignore_err',
        '-i', file_path,
        '-c:v', 'libx264',  # Re-encode video
        '-c:a', 'aac',      # Re-encode audio
        '-preset', 'fast',  # Más rápido
        '-y',
        output_path
    ]
    try:
        result = subprocess.run(cmd_encode, capture_output=True, text=True, timeout=600)
        if result.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print(f"Video reparado con FFmpeg (encode): {output_path}")
            return True
        else:
            print(f"FFmpeg encode falló para {file_path}: {result.stderr}")
    except subprocess.TimeoutExpired:
        print(f"Timeout en encode para {file_path}")
    except Exception as e:
        print(f"Error en encode para {file_path}: {e}")
    
    return False

def repair_video_opencv(file_path, output_dir):
    """Repara video usando OpenCV como fallback."""
    try:
        cap = cv2.VideoCapture(file_path)
        if not cap.isOpened():
            print(f"No se pudo abrir el video {file_path}")
            return

        # Obtener propiedades del video
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Determinar el codec basado en la extensión
        ext = os.path.splitext(file_path)[1].lower()
        if ext == '.mp4':
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        elif ext == '.avi':
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
        else:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Default a MP4

        output_path = os.path.join(output_dir, os.path.basename(file_path))
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            out.write(frame)
            frame_count += 1

        cap.release()
        out.release()

        if frame_count > 0:
            print(f"Video reparado con OpenCV: {output_path} ({frame_count} frames)")
        else:
            print(f"No se pudieron extraer frames del video {file_path}")
            if os.path.exists(output_path):
                os.remove(output_path)
    except Exception as e:
        print(f"Error al procesar el video {file_path}: {e}")

def repair_video(file_path, output_dir):
    """Intenta reparar video primero con FFmpeg, luego con OpenCV."""
    if check_ffmpeg():
        if repair_video_ffmpeg(file_path, output_dir):
            return
    # Fallback a OpenCV
    repair_video_opencv(file_path, output_dir)

def main(folder_path):
    """Escanea la carpeta y repara archivos corruptos en paralelo, guardando en una carpeta 'Reparados'."""
    if not os.path.isdir(folder_path):
        print(f"La ruta {folder_path} no es una carpeta válida.")
        return

    # Crear carpeta 'Reparados' dentro de la carpeta de entrada
    repaired_dir = os.path.join(folder_path, "Reparados")
    os.makedirs(repaired_dir, exist_ok=True)
    print(f"Archivos reparados se guardarán en: {repaired_dir}")

    # Recolectar archivos a procesar
    images = []
    videos = []
    for root, dirs, files in os.walk(folder_path):
        # Evitar procesar la carpeta 'Reparados' si ya existe
        if "Reparados" in dirs:
            dirs.remove("Reparados")

        for file in files:
            file_path = os.path.join(root, file)
            ext = os.path.splitext(file)[1].lower()
            if ext in IMAGE_EXTENSIONS:
                images.append(file_path)
            elif ext in VIDEO_EXTENSIONS:
                videos.append(file_path)

    # Procesar en paralelo usando ProcessPoolExecutor
    with ProcessPoolExecutor() as executor:
        print(f"Procesando {len(images)} imágenes en paralelo...")
        # Procesar imágenes
        futures = [executor.submit(repair_image, img, repaired_dir) for img in images]
        for future in futures:
            future.result()  # Esperar a que terminen

        print(f"Procesando {len(videos)} videos en paralelo...")
        # Procesar videos
        futures = [executor.submit(repair_video, vid, repaired_dir) for vid in videos]
        for future in futures:
            future.result()  # Esperar a que terminen

    print("Procesamiento completado.")

if __name__ == "__main__":
    if len(sys.argv) == 1:
        # Usar la carpeta "Todo" por defecto si no se pasa argumento
        folder = os.path.join(os.path.dirname(__file__), "Arreglar")
        print(f"Usando carpeta por defecto: {folder}")
    elif len(sys.argv) == 2:
        folder = sys.argv[1]
    else:
        print("Uso: python Recuperar.py [ruta_a_la_carpeta]")
        print("Si no se especifica, usa la carpeta 'Arreglar' en el directorio del script.")
        sys.exit(1)
    main(folder)

## 2. Organizarlos por fecha
En las fotos se usa el EXIF, en videos se usa FFmpeg

In [ ]:
import os
import shutil
import subprocess
from datetime import datetime
from PIL import Image, ExifTags

# -------------------------
# EXTENSIONES
# -------------------------
EXTS_IMG = {".jpg", ".jpeg", ".png", ".heic", ".bmp", ".gif"}
EXTS_VID = {".mp4", ".mov", ".avi", ".mkv", ".wmv"}
EXTS = EXTS_IMG | EXTS_VID

# -------------------------
# FECHAS PARA FOTOS
# -------------------------
def get_exif_date(path):
    """Intenta obtener la fecha EXIF de una foto."""
    try:
        img = Image.open(path)
        exif = img._getexif()
        if not exif:
            return None

        for tag, value in exif.items():
            name = ExifTags.TAGS.get(tag, tag)
            if name in ("DateTimeOriginal", "DateTimeDigitized"):
                return datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
    except:
        return None

# -------------------------
# FECHAS PARA VIDEOS (FFPROBE)
# -------------------------
def get_video_date_ffmpeg(path):
    """Intenta obtener la fecha real del video usando ffprobe."""
    try:
        cmd = [
            "ffprobe",
            "-v", "quiet",
            "-print_format", "json",
            "-show_entries", "format_tags=creation_time",
            path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        out = result.stdout

        if not out.strip():
            return None

        import json
        data = json.loads(out)
        tags = data.get("format", {}).get("tags", {})
        ts = tags.get("creation_time")
        if not ts:
            return None

        # Ejemplo: "2020-03-15T14:22:10.000000Z"
        ts = ts.replace("Z", "+00:00")
        return datetime.fromisoformat(ts)
    except:
        return None

# -------------------------
# FECHA DEL ARCHIVO (FALLBACK)
# -------------------------
def get_file_date(path):
    ts = os.path.getmtime(path)
    return datetime.fromtimestamp(ts)

# -------------------------
# SISTEMA INTELIGENTE DE FECHAS
# -------------------------
def get_best_date(path):
    ext = os.path.splitext(path)[1].lower()

    # Fotos → EXIF primero
    if ext in EXTS_IMG:
        d = get_exif_date(path)
        if d:
            return d

    # Videos → ffprobe primero
    if ext in EXTS_VID:
        d = get_video_date_ffmpeg(path)
        if d:
            return d

    # Fallback universal
    return get_file_date(path)

# -------------------------
# ORGANIZAR POR AÑO / AÑO-MES
# -------------------------
def organizar_por_anio_mes(carpeta_origen, carpeta_destino):
    for raiz, _, archivos in os.walk(carpeta_origen):
        for archivo in archivos:
            ext = os.path.splitext(archivo.lower())[1]
            if ext not in EXTS:
                continue

            ruta = os.path.join(raiz, archivo)
            fecha = get_best_date(ruta)

            anio = str(fecha.year)
            mes = f"{fecha.year}-{fecha.month:02d}"

            destino_mes = os.path.join(carpeta_destino, anio, mes)
            os.makedirs(destino_mes, exist_ok=True)

            destino_final = os.path.join(destino_mes, archivo)

            # Evitar sobrescribir archivos
            if os.path.exists(destino_final):
                base, ext = os.path.splitext(archivo)
                destino_final = os.path.join(destino_mes, f"{base}_dup{ext}")

            print(f"Moviendo: {ruta} → {destino_final}")
            shutil.move(ruta, destino_final)

    print("Organización completada.")

# -------------------------
# CONFIGURACIÓN DEL USUARIO
# -------------------------
if __name__ == "__main__":
    carpeta_origen = os.path.join(os.path.dirname(__file__), "Arreglar")
    carpeta_destino = os.path.join(os.path.dirname(__file__), "Fecha")

    organizar_por_anio_mes(carpeta_origen, carpeta_destino)

## 3. Mejorar la Calidad de Fotos y Videos

In [ ]:
import os
import cv2
import shutil
import subprocess
from tqdm import tqdm
import tempfile

# -----------------------------
# CONFIGURACIÓN
# -----------------------------
REALESRGAN = r"C:\Users\LuKGa\Downloads\realesrgan-ncnn-vulkan-20220424-windows\realesrgan-ncnn-vulkan.exe"

EXTS_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".heic"}
EXTS_VIDEO = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}

LOG_FILE = "errores_mejora.txt"


# -----------------------------
# DETECTAR MINIATURAS
# -----------------------------
def is_thumbnail(path, min_size=500):
    img = cv2.imread(path)
    if img is None:
        return True
    h, w = img.shape[:2]
    return w < min_size or h < min_size


# -----------------------------
# MEJORAR IMAGEN CON REINTENTOS
# -----------------------------
def enhance_with_retries(input_path, output_path, retries=3):
    for attempt in range(1, retries + 1):
        try:
            cmd = [
                REALESRGAN,
                "-i", input_path,
                "-o", output_path,
                "-n", "realesrgan-x4plus",
                "-s", "4"
            ]

            subprocess.run(cmd)

            if os.path.exists(output_path) and os.path.getsize(output_path) > 10000:
                return True

        except Exception:
            pass

    return False


# -----------------------------
# PROCESAR IMAGEN
# -----------------------------
def process_image(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if is_thumbnail(input_path):
        os.remove(input_path)
        return f"Miniatura eliminada: {input_path}"

    ok = enhance_with_retries(input_path, output_path)

    if ok:
        return f"Mejorada: {input_path}"
    else:
        shutil.copy2(input_path, output_path)
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(f"Error mejorando imagen: {input_path}\n")
        return f"Falló mejora, copiada: {input_path}"


# -----------------------------
# PROCESAR VIDEO
# -----------------------------
def process_video(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    temp_dir = tempfile.mkdtemp()
    frames_dir = os.path.join(temp_dir, "frames")
    enhanced_dir = os.path.join(temp_dir, "enhanced")
    os.makedirs(frames_dir, exist_ok=True)
    os.makedirs(enhanced_dir, exist_ok=True)

    # Extraer frames
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for i in tqdm(range(frame_count), desc=f"Extrayendo frames {os.path.basename(input_path)}"):
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(os.path.join(frames_dir, f"{i:06d}.png"), frame)

    cap.release()

    # Mejorar frames
    frame_files = sorted(os.listdir(frames_dir))

    for f in tqdm(frame_files, desc=f"Mejorando frames {os.path.basename(input_path)}"):
        inp = os.path.join(frames_dir, f)
        out = os.path.join(enhanced_dir, f)
        enhance_with_retries(inp, out)

    # Reconstruir video
    first_frame = cv2.imread(os.path.join(enhanced_dir, frame_files[0]))
    h, w = first_frame.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    for f in tqdm(frame_files, desc=f"Reconstruyendo video {os.path.basename(input_path)}"):
        frame = cv2.imread(os.path.join(enhanced_dir, f))
        video_out.write(frame)

    video_out.release()

    shutil.rmtree(temp_dir)

    return f"Video mejorado: {input_path}"


# -----------------------------
# PROCESAR TODA LA CARPETA
# -----------------------------
def run_folder(input_folder, output_folder):
    tasks = []

    for root, _, files in os.walk(input_folder):
        for f in files:
            ext = f.lower().split(".")[-1]
            ext = "." + ext

            input_path = os.path.join(root, f)
            relative = os.path.relpath(input_path, input_folder)
            output_path = os.path.join(output_folder, relative)

            base, ext2 = os.path.splitext(output_path)
            output_path = base + "_mejorado" + ext2

            if ext in EXTS_IMG:
                tasks.append(("img", input_path, output_path))
            elif ext in EXTS_VIDEO:
                tasks.append(("video", input_path, output_path))

    print(f"Total archivos encontrados: {len(tasks)}")

    for ttype, input_path, output_path in tqdm(tasks, desc="Procesando"):
        if ttype == "img":
            print(process_image(input_path, output_path))
        else:
            print(process_video(input_path, output_path))


# -----------------------------
# EJECUTAR
# -----------------------------
if __name__ == "__main__":
    run_folder(
        input_folder= os.path.join(os.path.dirname(__file__), "Fecha"),
        output_folder= os.path.join(os.path.dirname(__file__), "Mejorado")
    )